In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
df = pd.read_csv("../Data/c_w_data")

## 1. Data Setup

In [20]:
df.head()

,Series,Surface,Round,Score,dj_pts,dj_rank,dj_odds,opp_pts,opp_rank,opp_odds,dj_win,age,is_outdoor,is_five_sets,5SMA
0,International,Clay,2nd Round,6-4 6-2,417,97,5.00,1170,28,1.14,0,18,1,0,1.000000
1,Masters,Hard,1st Round,6-3 6-7 4-6,440,97,3.50,1385,18,1.28,0,18,1,0,0.500000
2,Grand Slam,Hard,1st Round,7-5 4-6 7-5 0-6 7-5,431,97,2.50,825,43,1.50,1,18,1,1,0.333333
3,Grand Slam,Hard,2nd Round,6-3 5-7 7-6 6-3,431,97,3.50,1230,24,1.28,1,18,1,1,0.500000
4,Grand Slam,Hard,3rd Round,1-6 6-4 7-6 4-6 4-6,431,97,2.75,770,48,1.39,0,18,1,1,0.600000


Firstly, we must use one-hot encoding to create categorical variables to use for our regression. Once we have the one hot encoded dataframe, we can drop the corresponding columns in the original dataframe, and then concatenate the one-hot dataframe with the original.

In [21]:
encoded = pd.concat([pd.get_dummies(df["Series"]).astype(int),
                    pd.get_dummies(df["Surface"]).astype(int),
                    pd.get_dummies(df["Round"]).astype(int)], axis=1)

In [22]:
encoded.head()

,ATP250,ATP500,Grand Slam,International,International Gold,Masters,Masters 1000,Masters Cup,Carpet,Clay,Grass,Hard,1st Round,2nd Round,3rd Round,4th Round,Quarterfinals,Round Robin,Semifinals,The Final
0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0


In [23]:
df = df.copy().drop(columns=['Series', 'Surface', 'Round'])

We must drop the Score and dj_win columns since these are only available for each match once we know the outcome. Keeping these in the dataframe for the linear regression will result in unwanted data leakage.

In [25]:
df = df.copy().drop(columns=['Score', 'dj_win'])

Now we form the prediction variable matrix and the target variable vector.

In [31]:
y = df['dj_odds'].to_numpy()
X = df[['dj_pts', 'dj_rank', 'opp_pts', 'opp_rank', 'opp_odds',
       'age', 'is_outdoor', 'is_five_sets', '5SMA']].to_numpy()

Now we shuffle our data so that our model doesn't learn memorise a specific time frame of the data, and then perform a train / test split.

In [45]:
def shuffle_data(X, y, seed=0):
    rng = np.random.default_rng(seed)
    indices = np.arange(X.shape[0])
    
    shuffled_indices = rng.permutation(indices)

    return X[shuffled_indices], y[shuffled_indices]

In [47]:
shuff_X, shuff_y = shuffle_data(X, y)

## 2. Least Squares Regression

In [38]:
theta = np.linalg.solve(X.T @ X, X.T @ y)
y_hat = X @ theta